# 01 - 像素与色彩深度：图像的微观世界

**Cambridge International AS & A Level Computer Science (9618) - Section 1.2**

---

> "你在屏幕上看到的每一张图片、每一个文字、每一个图标，都是由成千上万个微小的彩色小方块组成的。"

在这一节中，我们将从最基础的问题开始：
- 计算机是怎么"看到"图像的？
- 一个像素到底是什么？
- 为什么 8 位就是 256 种颜色？
- RGB 是怎么混合出千万种颜色的？

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 WenQuanYi Zen Hei 字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('WenQuanYi Zen Hei')
if 'WenQuanYi Zen Hei' in test_font or 'wqy' in test_font.lower():
    print('✅ 中文字体 WenQuanYi Zen Hei 已加载')
else:
    print(f'⚠️ WenQuanYi Zen Hei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 WenQuanYi Zen Hei 字体文件')

import sys
try:
    import matplotlib.patches as patches
    from matplotlib.colors import ListedColormap
    import numpy as np
    from IPython.display import display, HTML, Image
    print('Libraries loaded!')
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'matplotlib', 'numpy', '-q'])
    import matplotlib.patches as patches
    from matplotlib.colors import ListedColormap
    import numpy as np
    from IPython.display import display, HTML, Image
    print('Installed & loaded!')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10,6),
    'font.sans-serif': ['WenQuanYi Zen Hei','Noto Sans CJK SC','DejaVu Sans'], 'axes.unicode_minus': False})


---
## 1. 从现实世界到数字世界

### 1.1 人眼 vs 计算机

人眼看到的世界是 **连续的**（模拟信号）——颜色可以无限细分，过渡是平滑的。

但计算机只认识 **0 和 1**（数字信号）——一切信息都必须变成二进制数字。

所以问题来了：**怎么用 0 和 1 来表示一张图片？**

答案是：把图片切成无数个小格子，每个格子存一个颜色值。

### 1.2 像素 (Pixel) 的诞生

**Pixel = Picture Element（图像元素）**

想象你拿一张照片，用一个放大镜不断放大——最终你会看到一个个小方块。每个小方块就是一个 **像素**。

- 每个像素只有 **一种颜色**（不能半红半蓝）
- 像素越多，图像越细腻
- 像素越少，图像越"马赛克"

这就像用乐高积木拼图——积木越小越多，拼出来的图越逼真。

In [ ]:
# 直观演示：用不同数量的"像素"表示一个圆
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))

for ax, res in zip(axes, [3, 6, 12, 24, 64]):
    y, x = np.mgrid[-1:1:complex(res), -1:1:complex(res)]
    circle = (x**2 + y**2 < 0.7).astype(float)
    ax.imshow(circle, cmap='Blues', interpolation='nearest', vmin=0, vmax=1)
    ax.set_title(f'{res}x{res}\n({res*res} pixels)', fontsize=11, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    if res <= 12:
        ax.set_xticks(np.arange(-0.5, res, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, res, 1), minor=True)
        ax.grid(which='minor', color='gray', linewidth=0.5, alpha=0.5)

plt.suptitle('Same Circle, Different Number of Pixels', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('3x3=9 pixels:   You can barely tell it is a circle')
print('6x6=36 pixels:  Starting to look round')
print('64x64=4096:     Smooth! But 450x more data than 3x3!')
print()
print('This is the fundamental trade-off:')
print('  More pixels = Better quality = Larger file size')

---
## 2. 色彩深度：为什么 8 位 = 256 种颜色？

### 2.1 从开关说起

回想第一章学的内容：计算机里的一切都是二进制——0 和 1。

一个 bit 就像一个开关，只有两种状态：

| bit 值 | 状态 |
|:---:|:---:|
| 0 | 关 OFF |
| 1 | 开 ON |

如果我们用 **1 个 bit** 来表示一个像素的颜色：
- `0` = 黑色
- `1` = 白色
- **只有 2 种颜色**

### 2.2 增加 bit 数 = 指数增长！

如果我们用 **2 个 bit** 呢？

| bit 1 | bit 0 | 组合 | 颜色编号 |
|:---:|:---:|:---:|:---:|
| 0 | 0 | 00 | 颜色 0 |
| 0 | 1 | 01 | 颜色 1 |
| 1 | 0 | 10 | 颜色 2 |
| 1 | 1 | 11 | 颜色 3 |

2 个 bit = **4 种组合** = 4 种颜色

**规律：n 个 bit 能表示 2^n 种不同的值！**

这就是为什么：
- 1 bit = 2^1 = **2** 种颜色
- 2 bits = 2^2 = **4** 种颜色
- 3 bits = 2^3 = **8** 种颜色
- 4 bits = 2^4 = **16** 种颜色
- 8 bits = 2^8 = **256** 种颜色
- 16 bits = 2^16 = **65,536** 种颜色
- 24 bits = 2^24 = **16,777,216** 种颜色

**这不是巧合，这是二进制的数学本质！**

In [ ]:
# 交互演示：n 个 bit 能表示多少种颜色？
print('位数(n)  |  可能的组合数(2^n)  |  所有可能的二进制值')
print('-' * 65)

for n in range(1, 9):
    count = 2 ** n
    # 生成所有可能的二进制组合
    combos = [format(i, f'0{n}b') for i in range(count)]
    # 只显示前几个和最后一个
    if count <= 8:
        combo_str = ', '.join(combos)
    else:
        combo_str = ', '.join(combos[:4]) + f' ... {combos[-1]}'
    print(f'  {n} bit  |  2^{n} = {count:>5} colours  |  {combo_str}')

print()
print('Notice the pattern: each extra bit DOUBLES the number of colours!')
print('  1 bit:  2 colours')
print('  2 bits: 4 colours   (2 x 2)')
print('  3 bits: 8 colours   (2 x 2 x 2)')
print('  8 bits: 256 colours (2 x 2 x 2 x 2 x 2 x 2 x 2 x 2)')
print()
print('This is exponential growth - a core concept in computer science!')

In [ ]:
# 可视化：每多一个 bit，颜色数翻倍
fig, ax = plt.subplots(figsize=(12, 6))

bits_range = range(1, 25)
colours_count = [2**n for n in bits_range]

ax.bar(list(bits_range), colours_count, color=plt.cm.viridis(np.linspace(0, 1, 24)),
       edgecolor='black', linewidth=0.5)
ax.set_yscale('log')  # 对数刻度，否则看不清小的
ax.set_xlabel('Colour Depth (bits)', fontsize=13)
ax.set_ylabel('Number of Colours (log scale)', fontsize=13)
ax.set_title('Exponential Growth: Colour Depth vs Number of Colours', fontsize=15, fontweight='bold')

# 标注几个关键点
highlights = {1: '2', 4: '16', 8: '256', 16: '65K', 24: '16.7M'}
for b, label in highlights.items():
    ax.annotate(f'{b}-bit\n{label}', xy=(b, 2**b), xytext=(b, 2**b * 3),
                fontsize=10, fontweight='bold', ha='center',
                arrowprops=dict(arrowstyle='->', color='red'),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

ax.set_xticks(list(bits_range))
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## 3. RGB 色彩模型：三种光混出千万色

### 3.1 光的三原色

人眼有三种感光细胞，分别对 **红 (Red)**、**绿 (Green)**、**蓝 (Blue)** 敏感。

计算机利用这个原理：每个像素由 **R、G、B 三个通道** 组成，每个通道控制一种光的强度。

### 3.2 24位真彩色的原理

在 24 位色彩深度中：
- **红色通道：8 bits**（0-255 的强度）
- **绿色通道：8 bits**（0-255 的强度）
- **蓝色通道：8 bits**（0-255 的强度）

总共：8 + 8 + 8 = **24 bits per pixel**

每个通道 8 bits = 256 级强度，三个通道组合：
$$256 \times 256 \times 256 = 16,777,216 \text{ 种颜色}$$

这就是 **真彩色 (True Colour)** 的由来！

### 3.3 颜色编码举例

| 颜色 | R | G | B | 十六进制 |
|:---|:---:|:---:|:---:|:---:|
| 纯红 | 255 | 0 | 0 | #FF0000 |
| 纯绿 | 0 | 255 | 0 | #00FF00 |
| 纯蓝 | 0 | 0 | 255 | #0000FF |
| 白色 | 255 | 255 | 255 | #FFFFFF |
| 黑色 | 0 | 0 | 0 | #000000 |
| 黄色 | 255 | 255 | 0 | #FFFF00 |
| 紫色 | 128 | 0 | 128 | #800080 |

注意十六进制 FF = 十进制 255 = 二进制 11111111（刚好 8 个 1）

**这就是第一章学的十六进制在实际中的应用！** 网页设计中的颜色代码就是 RGB 的十六进制表示。

In [ ]:
# RGB 色彩混合交互演示
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# 上排：基本颜色
basic_colours = [
    ((255,0,0), 'Red\n#FF0000\nR=255,G=0,B=0'),
    ((0,255,0), 'Green\n#00FF00\nR=0,G=255,B=0'),
    ((0,0,255), 'Blue\n#0000FF\nR=0,G=0,B=255'),
    ((255,255,255), 'White\n#FFFFFF\nR=255,G=255,B=255'),
]

for ax, (rgb, label) in zip(axes[0], basic_colours):
    colour = np.array([[rgb]]) / 255.0
    ax.imshow(np.ones((50, 50, 3)) * colour)
    textcolor = 'white' if sum(rgb) < 400 else 'black'
    ax.text(25, 25, label, ha='center', va='center', fontsize=10,
            fontweight='bold', color=textcolor)
    ax.set_xticks([]); ax.set_yticks([])

# 下排：混合颜色
mixed_colours = [
    ((255,255,0), 'Yellow\nR+G\n#FFFF00'),
    ((255,0,255), 'Magenta\nR+B\n#FF00FF'),
    ((0,255,255), 'Cyan\nG+B\n#00FFFF'),
    ((255,128,0), 'Orange\nR=255,G=128\n#FF8000'),
]

for ax, (rgb, label) in zip(axes[1], mixed_colours):
    colour = np.array([[rgb]]) / 255.0
    ax.imshow(np.ones((50, 50, 3)) * colour)
    textcolor = 'white' if sum(rgb) < 400 else 'black'
    ax.text(25, 25, label, ha='center', va='center', fontsize=10,
            fontweight='bold', color=textcolor)
    ax.set_xticks([]); ax.set_yticks([])

axes[0][0].set_ylabel('Primary\nColours', fontsize=13, fontweight='bold')
axes[1][0].set_ylabel('Mixed\nColours', fontsize=13, fontweight='bold')

plt.suptitle('RGB Colour Mixing: 3 channels, each 0-255', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key insight: EVERY colour on your screen is a mix of R, G, B!')
print('Each channel has 256 levels (0-255), giving 256^3 = 16,777,216 total colours.')
print()
print('Web designers use HEX codes: #RRGGBB')
print('  FF in hex = 255 in decimal = 11111111 in binary (Chapter 1!)')

In [ ]:
# 交互工具：自定义 RGB 颜色
def show_colour(r, g, b):
    """Display a colour and its code representations"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))

    # Colour swatch
    colour = np.array([[[r/255, g/255, b/255]]])
    ax1.imshow(np.ones((80, 120, 3)) * colour)
    textcolor = 'white' if (r + g + b) < 400 else 'black'
    ax1.text(60, 40, f'R={r} G={g} B={b}', ha='center', va='center',
             fontsize=14, fontweight='bold', color=textcolor)
    ax1.set_xticks([]); ax1.set_yticks([])
    ax1.set_title('Your Colour', fontsize=14, fontweight='bold')

    # Channel breakdown
    channels = [('Red', r, '#FF0000'), ('Green', g, '#00FF00'), ('Blue', b, '#0000FF')]
    y_pos = [2, 1, 0]
    for y, (name, val, col) in zip(y_pos, channels):
        ax2.barh(y, val, color=col, height=0.6, alpha=0.7, edgecolor='black')
        ax2.text(val + 5, y, f'{val} ({bin(val)[2:].zfill(8)})', va='center', fontsize=11)
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(['Red', 'Green', 'Blue'], fontsize=12, fontweight='bold')
    ax2.set_xlim(0, 300)
    ax2.set_xlabel('Intensity (0-255)', fontsize=12)
    ax2.set_title('Channel Breakdown', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()

    hex_code = f'#{r:02X}{g:02X}{b:02X}'
    bits_total = 24
    print(f'Hex: {hex_code}')
    print(f'Binary: {bin(r)[2:].zfill(8)} {bin(g)[2:].zfill(8)} {bin(b)[2:].zfill(8)}')
    print(f'Total: {bits_total} bits = 3 bytes per pixel')

# Try these! Change the R, G, B values (0-255)
show_colour(255, 128, 0)   # Orange

# Try your own:
# show_colour(100, 200, 50)

In [ ]:
# 渐变色带：展示 RGB 通道如何工作
fig, axes = plt.subplots(4, 1, figsize=(14, 8))

x = np.linspace(0, 255, 256).astype(int)

# Red channel
red_strip = np.zeros((30, 256, 3))
red_strip[:, :, 0] = x / 255
axes[0].imshow(red_strip, aspect='auto')
axes[0].set_title('Red Channel: R varies 0-255, G=0, B=0', fontsize=12, fontweight='bold')
axes[0].set_ylabel('R', fontsize=14, color='red', fontweight='bold')

# Green channel
green_strip = np.zeros((30, 256, 3))
green_strip[:, :, 1] = x / 255
axes[1].imshow(green_strip, aspect='auto')
axes[1].set_title('Green Channel: R=0, G varies 0-255, B=0', fontsize=12, fontweight='bold')
axes[1].set_ylabel('G', fontsize=14, color='green', fontweight='bold')

# Blue channel
blue_strip = np.zeros((30, 256, 3))
blue_strip[:, :, 2] = x / 255
axes[2].imshow(blue_strip, aspect='auto')
axes[2].set_title('Blue Channel: R=0, G=0, B varies 0-255', fontsize=12, fontweight='bold')
axes[2].set_ylabel('B', fontsize=14, color='blue', fontweight='bold')

# All channels together (grayscale when equal)
gray_strip = np.zeros((30, 256, 3))
gray_strip[:, :, 0] = x / 255
gray_strip[:, :, 1] = x / 255
gray_strip[:, :, 2] = x / 255
axes[3].imshow(gray_strip, aspect='auto')
axes[3].set_title('All Equal (R=G=B): Grayscale from black to white', fontsize=12, fontweight='bold')
axes[3].set_ylabel('RGB', fontsize=14, fontweight='bold')

for ax in axes:
    ax.set_xticks([0, 64, 128, 192, 255])
    ax.set_xticklabels(['0', '64', '128', '192', '255'])
    ax.set_yticks([])

plt.suptitle('How RGB Channels Create Colours', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('Each channel is like a dimmer switch: 0 = off, 255 = full brightness')
print('When R=G=B, you get shades of gray')
print('When all are 255, you get white; when all are 0, you get black')

---
## 4. 色彩深度对比实验

现在你理解了原理，让我们亲眼看看不同色彩深度的效果：

In [ ]:
# 色彩深度对比：从 1 位到 24 位
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# Create a rich test image
y, x = np.mgrid[-1:1:200j, -1:1:200j]
r = np.sqrt(x**2 + y**2)
img = np.stack([
    np.clip((1 - r) * 255, 0, 255),       # Red: bright in center
    np.clip(np.abs(x) * 255, 0, 255),      # Green: bright on sides
    np.clip(np.abs(y) * 255, 0, 255),       # Blue: bright top/bottom
], axis=-1).astype(np.uint8)

configs = [
    (1, '1-bit\n2 colours', '1 bit/pixel = only B&W'),
    (2, '2-bit\n4 colours', '2 bits/pixel = 4 colour levels'),
    (3, '3-bit\n8 colours', '3 bits/pixel = 8 colours'),
    (4, '4-bit\n16 colours', '4 bits/pixel = basic colour'),
    (5, '5-bit\n32 colours', '5 bits/pixel'),
    (6, '6-bit\n64 colours', '6 bits/pixel = getting better'),
    (7, '7-bit\n128 colours', '7 bits/pixel'),
    (8, '8-bit/ch\n16.7M colours', '8 bits/channel = True Colour!'),
]

for ax, (bits, title, desc) in zip(axes.flat, configs):
    n_levels = 2 ** bits
    if bits == 8:
        ax.imshow(img)
    else:
        q = (np.floor(img / 256.0 * n_levels) * (255 / max(n_levels - 1, 1))).astype(np.uint8)
        ax.imshow(q)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel(desc, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Colour Depth Comparison: From 1-bit to True Colour',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('Notice:')
print('  1-4 bits: Colours look "banded" - you can see sharp transitions')
print('  5-7 bits: Getting smoother, but still visible steps')
print('  8 bits per channel (24-bit): Smooth gradients, no visible banding')
print()
print('The human eye can distinguish about 10 million colours,')
print('so 24-bit (16.7 million) is usually enough for photo-realistic images.')

---
## 5. 32 位色彩：Alpha 通道

你可能听过"32位色彩"——比24位多了什么？

多了一个 **Alpha 通道（透明度）**：

| 通道 | 位数 | 范围 | 含义 |
|:---|:---:|:---:|:---|
| Red | 8 bits | 0-255 | 红色强度 |
| Green | 8 bits | 0-255 | 绿色强度 |
| Blue | 8 bits | 0-255 | 蓝色强度 |
| **Alpha** | 8 bits | 0-255 | **透明度** (0=完全透明, 255=完全不透明) |

**总计: 8+8+8+8 = 32 bits per pixel**

这就是为什么 PNG 格式的图片可以有透明背景，而 JPEG 不行——JPEG 只支持 24 位 (没有 Alpha)。

In [ ]:
# Alpha 通道演示
fig, axes = plt.subplots(1, 5, figsize=(16, 3))

# Checkerboard background (to show transparency)
check = np.zeros((50, 50, 3))
for i in range(50):
    for j in range(50):
        if (i // 5 + j // 5) % 2 == 0:
            check[i, j] = [0.8, 0.8, 0.8]
        else:
            check[i, j] = [1, 1, 1]

alphas = [255, 192, 128, 64, 0]
for ax, alpha in zip(axes, alphas):
    # Blend red square with checkerboard based on alpha
    blend = check.copy()
    a = alpha / 255
    blend[:, :, 0] = a * 1.0 + (1-a) * check[:, :, 0]  # Red
    blend[:, :, 1] = a * 0.0 + (1-a) * check[:, :, 1]  # Green
    blend[:, :, 2] = a * 0.0 + (1-a) * check[:, :, 2]  # Blue

    ax.imshow(blend)
    ax.set_title(f'Alpha = {alpha}\n({alpha/255*100:.0f}% opaque)', fontsize=11, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Alpha Channel: Controlling Transparency', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('Alpha = 255: Fully opaque (you cannot see through)')
print('Alpha = 128: Half transparent')
print('Alpha = 0:   Fully transparent (invisible!)')
print()
print('This is why PNG supports transparency but JPEG does not.')
print('PNG uses 32-bit (RGBA), JPEG uses 24-bit (RGB only).')

---
## 6. 总结与练习

### 核心知识点回顾

| 概念 | 要点 |
|:---|:---|
| **像素** | 图像的最小单元，每个像素存储一个颜色 |
| **色彩深度** | 每个像素用多少 bit 存颜色 |
| **2^n 规则** | n 个 bit 能表示 2^n 种颜色（二进制指数增长） |
| **RGB** | 三原色模型，每通道 8 bit = 24 bit 真彩色 |
| **24 位** | 8+8+8 = 256^3 = 16,777,216 种颜色 |
| **32 位** | 24 位 RGB + 8 位 Alpha 透明通道 |
| **HEX 颜色码** | #RRGGBB，每两位十六进制 = 一个通道 |

### 练习题

**基础题：**
1. 为什么 3 个 bit 能表示 8 种颜色？列出所有 3 位二进制组合。
2. 一个像素用 4 位色彩深度，能表示多少种颜色？
3. 颜色 #00FF80 的 R、G、B 值分别是多少（十进制）？
4. 为什么 R=G=B 时看到的是灰色？

**进阶题：**
5. 如果我们要表示 1000 种颜色，至少需要多少 bit？（提示：2^9 = 512, 2^10 = 1024）
6. 一个 100x100 像素的图片，24位色彩，需要多少字节存储？
7. 解释为什么 JPEG 不支持透明背景而 PNG 支持。
8. 如果计算机只有 1 位色彩深度，它能显示照片吗？效果会怎样？

In [ ]:
# 练习题答案

print('=== 基础题 ===')
print()
print('Q1: 3 bits, all combinations:')
for i in range(8):
    print(f'  {i:03b} = colour {i}')
print(f'  Total: 2^3 = {2**3} combinations')

print(f'\nQ2: 4 bits = 2^4 = {2**4} colours')

print(f'\nQ3: #00FF80:')
print(f'  R = 00 (hex) = {int("00", 16)} (decimal)')
print(f'  G = FF (hex) = {int("FF", 16)} (decimal)')
print(f'  B = 80 (hex) = {int("80", 16)} (decimal)')

print(f'\nQ4: When R=G=B, all three colour channels have equal intensity.')
print('  This means no colour dominates, producing neutral gray.')
print('  R=G=B=0 is black, R=G=B=255 is white, anything between is gray.')

print(f'\n=== Advanced ===')
print(f'\nQ5: 2^9=512 (not enough), 2^10=1024 (enough!), so minimum 10 bits.')

print(f'\nQ6: 100 x 100 x 24 bits = {100*100*24} bits = {100*100*24//8} bytes = {100*100*24//8/1000} KB')

print(f'\nQ7: JPEG uses 24-bit (RGB only, no alpha channel).')
print('  PNG uses 32-bit (RGBA, includes alpha for transparency).')
print('  Without alpha channel, there is no way to mark pixels as transparent.')

print(f'\nQ8: With 1-bit colour, each pixel is either black or white.')
print('  A photo would lose all colour and most detail,')
print('  becoming a high-contrast black-and-white silhouette.')
print('  You could recognise shapes, but no shading or colour.')

---
**Next:** [02 - 分辨率与文件大小计算](02_分辨率与文件大小.ipynb)